# Figuras do Modelo Final (sem retreinar)

Regenera as figuras de avaliação e de explicabilidade do modelo escolhido para o app (padrão:
**EfficientNetV2B0 × BRACOL**) a partir do `.keras` já treinado. As figuras originais do `mobilenet.ipynb` se perderam
quando a sessão do Kaggle terminou; aqui elas são refeitas **com o mesmo código** (e com o Grad-CAM corrigido).

| Figura / tabela | Arquivo em `outputs/{Modelo}_{dataset}/` |
|---|---|
| Matriz de confusão (absoluta e normalizada) + relatório por classe | `matriz_confusao_*.png`, `relatorio_classificacao_*.csv` |
| Curvas ROC e Precision-Recall, F1 por classe | `curvas_roc_pr_*.png`, `f1_por_classe_*.png` |
| Calibração (ECE) antes e depois do Temperature Scaling | `calibracao_*.png`, `calibracao_temperature_scaling_*.png` |
| Predição seletiva (risco × cobertura) | `risco_cobertura_*.png`, `abstencao_*.csv` |
| Grad-CAM (acertos e erros por classe), LIME, Integrated Gradients | `gradcam_*.png`, `lime_*.png`, `ig_*.png` |
| Painel comparativo das três explicações (uma linha por classe) | `painel_xai_*.png` |
| Latência e tamanho em TFLite de **todos** os `.keras` anexados | `outputs/latencia_tflite.csv` |

**Não dá para refazer sem treinar:** as curvas de aprendizado (precisam do `history` do treino) e o histórico do
Optuna (precisa do banco SQLite do estudo).

**Pré-requisitos:** anexe o(s) `.keras` (Add Input → Models/Datasets) e o dataset (BRACOL). Use o mesmo
acelerador do treino original (GPU) para que a latência fique comparável com a tabela.

In [ ]:
!pip install -q lime==0.2.0.1

import os
import glob
import time
import zipfile

import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.metrics import (auc, average_precision_score, classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, roc_curve)
from sklearn.preprocessing import label_binarize

for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print(f"TensorFlow: {tf.__version__}")

# ── Configuração ────────────────────────────────────────────────────────────────────
MODELO_FIG  = 'EfficientNetV2B0'
DATASET_FIG = 'bracol'
NOME_ARQUIVO = f'{MODELO_FIG}_{DATASET_FIG}'
OUTPUT_DIR = os.path.join('outputs', NOME_ARQUIVO)
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
IMG_SIZE = 224
N_POR_CLASSE = 3       # imagens por categoria no Grad-CAM (acertos e erros), como no mobilenet.ipynb
N_LIME_SAMPLES = 1000
MEDIR_LATENCIA = True  # benchmark TFLite de todos os .keras anexados (preenche a coluna da tabela comparativa)

# Mesmos caminhos do mobilenet.ipynb; se não existirem, o notebook procura em /kaggle/input uma pasta com
# train/val/test com o número de classes do modelo.
DATASETS = {
    'bracol':               '/kaggle/input/newdataset/new_dataset',
    'rocole':               '/kaggle/working/datasets/rocole',
    'coffee_leaf_diseases': '/kaggle/working/datasets/coffee_leaf_diseases',
    'jmuben':               '/kaggle/working/datasets/jmuben',
}

MODELOS = sorted(set(
    glob.glob('/kaggle/input/**/*.keras', recursive=True) + glob.glob('outputs/*/*.keras')
))
print(f"{len(MODELOS)} modelo(s) encontrado(s):")
for m in MODELOS:
    print(f"  {m}")

candidatos = [m for m in MODELOS if os.path.basename(m) == f'{NOME_ARQUIVO}.keras']
if not candidatos:
    raise FileNotFoundError(f"{NOME_ARQUIVO}.keras não encontrado. Anexe o modelo (Add Input → Models/Datasets).")
model = tf.keras.models.load_model(candidatos[0], compile=False)
print(f"\nModelo das figuras: {candidatos[0]} | entrada {model.input_shape} | saída {model.output_shape}")

In [ ]:
def _n_classes(pasta):
    return sum(os.path.isdir(os.path.join(pasta, d)) for d in os.listdir(pasta))


def resolver_splits(dataset, n_classes):
    """Pastas train/val/test: usa DATASETS se existir; senão procura em /kaggle/input."""
    base = DATASETS.get(dataset)
    if base and all(os.path.isdir(os.path.join(base, s)) for s in ('train', 'val', 'test')):
        return {s: os.path.join(base, s) for s in ('train', 'val', 'test')}
    for train in sorted(glob.glob('/kaggle/input/**/train', recursive=True)):
        raiz = os.path.dirname(train)
        if all(os.path.isdir(os.path.join(raiz, s)) for s in ('val', 'test')) and _n_classes(train) == n_classes:
            return {s: os.path.join(raiz, s) for s in ('train', 'val', 'test')}
    raise FileNotFoundError(f"train/val/test de '{dataset}' com {n_classes} classes não encontrado em /kaggle/input.")


def carregar_split(pasta):
    return tf.keras.utils.image_dataset_from_directory(
        pasta, image_size=(IMG_SIZE, IMG_SIZE), batch_size=32, label_mode='int', shuffle=False
    )


splits = resolver_splits(DATASET_FIG, model.output_shape[-1])
print(f"Dataset: {os.path.dirname(splits['test'])}")
val_dataset = carregar_split(splits['val'])
test_dataset = carregar_split(splits['test'])
class_names_geral = test_dataset.class_names
n_classes = len(class_names_geral)

# Funções auxiliares

In [ ]:
# Extrator até o mapa de características do backbone (entrada da 'global_avg_pool', com o pré-processamento
# incluído) e a cabeça densa sem a softmax -- os mesmos usados no Grad-CAM corrigido e no exportar_app.ipynb.
extrator = tf.keras.Model(model.inputs, model.get_layer('global_avg_pool').input)
idx_gap = model.layers.index(model.get_layer('global_avg_pool'))
camadas_cabeca = model.layers[idx_gap:-1]
W_final, b_final = [tf.constant(v, tf.float32) for v in model.get_layer('predicao_final').get_weights()]


def penultima_de(feature_maps):
    out = feature_maps
    for layer in camadas_cabeca:
        out = layer(out, training=False)
    return tf.cast(out, tf.float32)


def obter_logits(dataset):
    partes = []
    for batch_x, _ in dataset:
        partes.append((penultima_de(extrator(batch_x, training=False)) @ W_final + b_final).numpy())
    return np.concatenate(partes, axis=0)


def make_gradcam_heatmap(img_array, pred_index=None):
    """Grad-CAM corrigido (idêntico ao do mobilenet.ipynb): training=False explícito e derivada estável da softmax."""
    with tf.GradientTape() as tape:
        feature_maps = extrator(img_array, training=False)
        tape.watch(feature_maps)
        penultima = penultima_de(feature_maps)

    p = tf.nn.softmax(penultima @ W_final + b_final)[0]
    if pred_index is None:
        pred_index = int(tf.argmax(p))
    W_t = tf.transpose(W_final)
    dp_dpenultima = p[pred_index] * tf.reduce_sum(p[:, None] * (W_t[pred_index][None, :] - W_t), axis=0)
    grads = tape.gradient(penultima, feature_maps, output_gradients=dp_dpenultima[None, :])

    pooled_grads = tf.reduce_mean(tf.cast(grads, tf.float32), axis=(0, 1, 2))
    heatmap = tf.squeeze(tf.cast(feature_maps[0], tf.float32) @ pooled_grads[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-12)
    return heatmap.numpy()


def carregar_imagem(caminho):
    """Mesmo decode + resize bilinear do image_dataset_from_directory -> (uint8 para exibir, float32 com batch)."""
    img = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bilinear').numpy().astype(np.float32)
    return np.clip(np.round(img), 0, 255).astype(np.uint8), img[None]


def sobrepor(img_rgb, mapa, cmap='jet', alpha=0.45):
    mapa = tf.image.resize(mapa[..., None], img_rgb.shape[:2], method='bilinear').numpy()[..., 0]
    cores = plt.colormaps.get_cmap(cmap)(np.clip(mapa, 0, 1))[..., :3]
    return np.clip(cores * alpha + img_rgb.astype(np.float32) / 255 * (1 - alpha), 0, 1)


def salvar(nome_png, dpi=150):
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, nome_png), dpi=dpi, bbox_inches='tight')
    plt.show()
    plt.close()

# Predições no Teste e Matriz de Confusão

In [ ]:
y_true_test = np.concatenate([y.numpy() for _, y in test_dataset])
y_pred_probs_test = model.predict(test_dataset, verbose=0)
y_pred_test = y_pred_probs_test.argmax(1)
print(f"Teste: {len(y_true_test)} imagens | acurácia {np.mean(y_true_test == y_pred_test):.4f} | "
      f"F1-Macro {f1_score(y_true_test, y_pred_test, average='macro'):.4f}")

cm = confusion_matrix(y_true_test, y_pred_test)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, matriz, fmt, titulo in [(axes[0], cm, 'd', 'Contagem'), (axes[1], cm_norm, '.2f', 'Normalizada por classe real')]:
    sns.heatmap(matriz, annot=True, fmt=fmt, cmap='Blues', xticklabels=class_names_geral,
                yticklabels=class_names_geral, linewidths=.5, cbar_kws={'shrink': .75}, ax=ax)
    ax.set_title(f'Matriz de Confusão — {MODELO_FIG} / {DATASET_FIG} (Teste) — {titulo}', fontsize=13, pad=15)
    ax.set_ylabel('Classe Verdadeira (Real)', fontsize=12)
    ax.set_xlabel('Classe Predita (Modelo)', fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
salvar(f'matriz_confusao_{NOME_ARQUIVO}.png')

print(classification_report(y_true_test, y_pred_test, target_names=class_names_geral, digits=4))
pd.DataFrame(classification_report(y_true_test, y_pred_test, target_names=class_names_geral, output_dict=True)) \
    .T.round(4).to_csv(os.path.join(OUTPUT_DIR, f'relatorio_classificacao_{NOME_ARQUIVO}.csv'))

# Curvas ROC, Precision-Recall e F1 por Classe

In [ ]:
y_true_bin = label_binarize(y_true_test, classes=list(range(n_classes)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, class_name in enumerate(class_names_geral):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs_test[:, i])
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{class_name} (AUC={auc(fpr, tpr):.3f})')
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs_test[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_pred_probs_test[:, i])
    axes[1].plot(recall, precision, linewidth=2, label=f'{class_name} (AP={ap:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatório')
axes[0].set_title('Curvas ROC por Classe (One-vs-Rest)', fontsize=13)
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[1].set_title('Curvas Precision-Recall por Classe', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left', fontsize=9)
axes[1].grid(True, alpha=0.3)
salvar(f'curvas_roc_pr_{NOME_ARQUIVO}.png')

f1_por_classe = f1_score(y_true_test, y_pred_test, average=None)
f1_macro = f1_score(y_true_test, y_pred_test, average='macro')
plt.figure(figsize=(10, 5))
cores = plt.cm.Blues(np.linspace(0.4, 0.85, n_classes))
bars = plt.bar(class_names_geral, f1_por_classe, color=cores, edgecolor='white', linewidth=0.8)
plt.axhline(y=f1_macro, color='red', linestyle='--', linewidth=1.8, label=f'F1-Macro = {f1_macro:.3f}')
for bar, val in zip(bars, f1_por_classe):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015, f'{val:.3f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('F1-Score por Classe no Conjunto de Teste', fontsize=14)
plt.xlabel('Classe', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.ylim(0, 1.12)
plt.legend(fontsize=11)
plt.xticks(rotation=30, ha='right')
salvar(f'f1_por_classe_{NOME_ARQUIVO}.png')

# Calibração de Confiança e Temperature Scaling

In [ ]:
def calcular_calibracao(y_true, y_pred, y_pred_probs, n_bins=10):
    confidences = y_pred_probs[np.arange(len(y_pred)), y_pred]
    corretos = (y_pred == y_true).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(confidences, bin_edges[1:-1]), 0, n_bins - 1)
    registros, ece = [], 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not mask.any():
            continue
        conf_media, acc_media = float(confidences[mask].mean()), float(corretos[mask].mean())
        ece += mask.mean() * abs(acc_media - conf_media)
        registros.append({'faixa_confianca': f'{bin_edges[b]:.1f}-{bin_edges[b + 1]:.1f}',
                          'confianca_media': conf_media, 'acuracia_media': acc_media, 'n_amostras': int(mask.sum())})
    return pd.DataFrame(registros), ece


def calcular_nll(logits, labels, T):
    z = logits / T
    z = z - z.max(axis=1, keepdims=True)
    probs = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
    return -np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)))


def softmax_T(logits, T):
    z = logits / T
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


df_calibracao, ece_valor = calcular_calibracao(y_true_test, y_pred_test, y_pred_probs_test)

# Temperatura ajustada no conjunto de VALIDAÇÃO (nunca no teste)
logits_val = obter_logits(val_dataset)
y_true_val = np.concatenate([y.numpy() for _, y in val_dataset])
T = minimize_scalar(lambda t: calcular_nll(logits_val, y_true_val, t), bounds=(0.05, 10.0), method='bounded').x
logits_test = obter_logits(test_dataset)
# Os logits reconstruídos têm que reproduzir o model.predict (garante que o training=True gravado no backbone
# pelo create_model não está ativo aqui)
assert np.allclose(softmax_T(logits_test, 1.0), y_pred_probs_test, atol=1e-4), "logits não batem com model.predict"
y_pred_probs_calibrado = softmax_T(logits_test, T)
assert np.array_equal(y_pred_probs_calibrado.argmax(1), y_pred_test), "Temperature scaling não deveria mudar o argmax"
df_calibracao_pos, ece_pos = calcular_calibracao(y_true_test, y_pred_test, y_pred_probs_calibrado)
print(f"Temperatura T = {T:.4f} | ECE antes {ece_valor:.4f} | ECE depois {ece_pos:.4f}")
print(df_calibracao_pos.to_string(index=False))

# Reliability diagram + histograma de confiança (antes)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibração perfeita')
axes[0].bar(df_calibracao['confianca_media'], df_calibracao['acuracia_media'], width=0.08, alpha=0.75,
            edgecolor='black', color='steelblue', label='Modelo')
axes[0].set_xlabel('Confiança média (probabilidade prevista)')
axes[0].set_ylabel('Acurácia real (fração de acertos)')
axes[0].set_title(f'Reliability Diagram — {MODELO_FIG} — ECE={ece_valor:.4f}', fontsize=12)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].hist(y_pred_probs_test[np.arange(len(y_pred_test)), y_pred_test], bins=20, color='steelblue',
             edgecolor='black', alpha=0.75)
axes[1].set_xlabel('Confiança prevista (probabilidade da classe predita)')
axes[1].set_ylabel('Número de amostras')
axes[1].set_title('Distribuição de Confiança no Teste', fontsize=12)
axes[1].grid(True, alpha=0.3)
salvar(f'calibracao_{NOME_ARQUIVO}.png')

# Antes x depois
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_plot, ece_plot, titulo in [(axes[0], df_calibracao, ece_valor, 'Antes (sem calibração)'),
                                      (axes[1], df_calibracao_pos, ece_pos, f'Depois (T={T:.2f})')]:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibração perfeita')
    ax.bar(df_plot['confianca_media'], df_plot['acuracia_media'], width=0.08, alpha=0.75, edgecolor='black',
           color='steelblue', label='Modelo')
    ax.set_xlabel('Confiança média')
    ax.set_ylabel('Acurácia real')
    ax.set_title(f'{titulo} — ECE={ece_plot:.4f}', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)
salvar(f'calibracao_temperature_scaling_{NOME_ARQUIVO}.png')

# Predição Seletiva (Abstenção por Confiança Calibrada)

In [ ]:
confidences_cal = y_pred_probs_calibrado[np.arange(len(y_pred_test)), y_pred_test]
corretos_bin = (y_pred_test == y_true_test)

limiares_curva = np.arange(0.0, 1.001, 0.01)
coberturas = [np.mean(confidences_cal >= t) for t in limiares_curva]
acuracias_mantidas = [corretos_bin[confidences_cal >= t].mean() if (confidences_cal >= t).any() else np.nan
                      for t in limiares_curva]

plt.figure(figsize=(8, 5))
plt.plot(coberturas, acuracias_mantidas, linewidth=2, color='steelblue')
plt.axhline(y=corretos_bin.mean(), color='gray', linestyle='--', alpha=0.6,
            label=f'Acurácia sem abstenção = {corretos_bin.mean():.3f}')
plt.xlabel('Cobertura (fração de imagens respondidas)')
plt.ylabel('Acurácia nas respostas mantidas')
plt.title(f'Curva Risco × Cobertura — {MODELO_FIG} (confiança calibrada)', fontsize=13)
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, alpha=0.3)
salvar(f'risco_cobertura_{NOME_ARQUIVO}.png')

registros_abstencao = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    mask = confidences_cal >= t
    registros_abstencao.append({
        'limiar': t,
        'cobertura (%)': round(100 * mask.mean(), 1),
        'acurácia nas mantidas': round(float(corretos_bin[mask].mean()), 4) if mask.any() else None,
        'n_abstidas': int((~mask).sum()),
        'erros_evitados': int(((~mask) & (~corretos_bin)).sum()),
        'erros_que_ainda_passam': int((mask & (~corretos_bin)).sum()),
    })
df_abstencao = pd.DataFrame(registros_abstencao)
print(df_abstencao.to_string(index=False))
df_abstencao.to_csv(os.path.join(OUTPUT_DIR, f'abstencao_{NOME_ARQUIVO}.csv'), index=False)

# Grad-CAM — Acertos e Erros por Classe

In [ ]:
# Mesma ordem do test_dataset (shuffle=False)
all_test_info = [{'path': p, 'true_label': int(t), 'pred_label': int(pr), 'confidence': float(conf)}
                 for p, t, pr, conf in zip(test_dataset.file_paths, y_true_test, y_pred_test,
                                           y_pred_probs_test[np.arange(len(y_pred_test)), y_pred_test])]

for class_idx, class_name in enumerate(class_names_geral):
    acertos = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx]
    erros = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] != class_idx]
    print(f"[{class_name}] Acertos: {len(acertos)} | Erros: {len(erros)}")

    fig, axes = plt.subplots(2, N_POR_CLASSE, figsize=(5 * N_POR_CLASSE, 9))
    fig.suptitle(f'Grad-CAM — {MODELO_FIG}/{DATASET_FIG} — Classe: {class_name}', fontsize=14, fontweight='bold')
    for row, (rotulo, samples) in enumerate([('Acertos (TP)', acertos[:N_POR_CLASSE]), ('Erros (FN)', erros[:N_POR_CLASSE])]):
        for col in range(N_POR_CLASSE):
            ax = axes[row, col]
            if col < len(samples):
                d = samples[col]
                img_rgb, img_array = carregar_imagem(d['path'])
                ax.imshow(sobrepor(img_rgb, make_gradcam_heatmap(img_array)))  # mapa da classe PREDITA
                ax.set_title(f"{rotulo}\nPred: {class_names_geral[d['pred_label']]} ({d['confidence'] * 100:.1f}%)", fontsize=8)
            else:
                ax.set_facecolor('#eeeeee')
            ax.axis('off')
    salvar(f'gradcam_{NOME_ARQUIVO}_{class_name}.png', dpi=120)

# LIME (Superpixels)

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)
explicacoes_lime = {}

for class_idx, class_name in enumerate(class_names_geral):
    acertos = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx]
    if not acertos:
        print(f"  [{class_name}] Sem acertos — pulando.")
        continue
    img_rgb, _ = carregar_imagem(acertos[0]['path'])
    explanation = explainer_lime.explain_instance(
        img_rgb, lambda imgs: model.predict(imgs.astype(np.float32), verbose=0),
        labels=(class_idx,), top_labels=None, hide_color=0, num_samples=N_LIME_SAMPLES, random_seed=SEED,
    )
    explicacoes_lime[class_idx] = explanation
    temp_pos, mask_pos = explanation.get_image_and_mask(class_idx, positive_only=True, num_features=6, hide_rest=False)
    temp_all, mask_all = explanation.get_image_and_mask(class_idx, positive_only=False, num_features=10, hide_rest=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f'Original\n{class_name}', fontsize=10)
    axes[1].imshow(mark_boundaries(temp_pos.astype(np.float32) / 255, mask_pos))
    axes[1].set_title('LIME — Regiões positivas\n(verde = favor da classe)', fontsize=10)
    axes[2].imshow(mark_boundaries(temp_all.astype(np.float32) / 255, mask_all))
    axes[2].set_title('LIME — Positivo + Negativo\n(verde = favor | vermelho = contra)', fontsize=10)
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'Explicação LIME — {MODELO_FIG}/{DATASET_FIG} — {class_name}', fontsize=13, fontweight='bold')
    salvar(f'lime_{NOME_ARQUIVO}_{class_name}.png', dpi=120)
    print(f"  [{class_name}] OK")

# Integrated Gradients

In [ ]:
def integrated_gradients(img_array, class_idx, m_steps=50, batch_size=32):
    """Integrated Gradients (Sundararajan et al., 2017), baseline preto, regra do trapézio -- igual ao mobilenet.ipynb."""
    img_array = tf.constant(img_array, tf.float32)
    baseline = tf.zeros_like(img_array)
    alphas = tf.linspace(0.0, 1.0, m_steps + 1)
    interp = tf.concat([baseline + a * (img_array - baseline) for a in alphas], axis=0)
    all_grads = []
    for i in range(0, len(interp), batch_size):
        batch = interp[i:i + batch_size]
        with tf.GradientTape() as tape:
            tape.watch(batch)
            target = model(batch, training=False)[:, class_idx]
        all_grads.append(tape.gradient(target, batch))
    grads = tf.concat(all_grads, axis=0)
    avg_grads = tf.reduce_mean((grads[:-1] + grads[1:]) / 2.0, axis=0, keepdims=True)
    return ((img_array - baseline) * avg_grads).numpy()[0]


mapas_ig = {}
for class_idx, class_name in enumerate(class_names_geral):
    acertos = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx]
    if not acertos:
        continue
    img_rgb, img_array = carregar_imagem(acertos[0]['path'])
    ig = integrated_gradients(img_array, class_idx)
    ig_abs = np.abs(ig).mean(axis=-1)
    ig_norm = (ig_abs - ig_abs.min()) / (ig_abs.max() - ig_abs.min() + 1e-8)
    mapas_ig[class_idx] = ig_norm
    ig_signed = ig.mean(axis=-1)
    ig_signed_norm = (ig_signed - ig_signed.min()) / (ig_signed.max() - ig_signed.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f'Original\n{class_name}', fontsize=10)
    im1 = axes[1].imshow(ig_norm, cmap='hot')
    axes[1].set_title('IG — Importância absoluta\n(mais quente = mais relevante)', fontsize=10)
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    axes[2].imshow(sobrepor(img_rgb, ig_signed_norm, cmap='RdBu_r', alpha=0.5))
    axes[2].set_title('IG — Overlay (azul = contra | vermelho = favor)', fontsize=10)
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'Integrated Gradients — {MODELO_FIG}/{DATASET_FIG} — {class_name}', fontsize=13, fontweight='bold')
    salvar(f'ig_{NOME_ARQUIVO}_{class_name}.png', dpi=120)
    print(f"  [{class_name}] OK")

# Painel Comparativo das Explicações

Uma linha por classe, **a mesma imagem** nas quatro colunas (o primeiro acerto de cada classe, o mesmo usado no LIME e
no IG): original, Grad-CAM, LIME (peso de cada superpixel, só os positivos) e Integrated Gradients. É a figura mais
direta para o texto comparar o que cada método destaca.

In [ ]:
classes_painel = [c for c in range(n_classes) if c in explicacoes_lime and c in mapas_ig]
fig, axes = plt.subplots(len(classes_painel), 4, figsize=(16, 4 * len(classes_painel)), squeeze=False)
for linha, class_idx in zip(axes, classes_painel):
    d = next(d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx)
    img_rgb, img_array = carregar_imagem(d['path'])

    exp = explicacoes_lime[class_idx]
    pesos = dict(exp.local_exp[class_idx])
    mapa_lime = np.vectorize(lambda s: max(pesos.get(s, 0.0), 0.0))(exp.segments).astype(np.float32)
    mapa_lime /= mapa_lime.max() + 1e-12

    paineis = [
        (img_rgb.astype(np.float32) / 255, f"{class_names_geral[class_idx]}\nconfiança {d['confidence'] * 100:.1f}%"),
        (sobrepor(img_rgb, make_gradcam_heatmap(img_array, class_idx)), 'Grad-CAM'),
        (sobrepor(img_rgb, mapa_lime), 'LIME (superpixels positivos)'),
        (sobrepor(img_rgb, mapas_ig[class_idx], cmap='hot', alpha=0.55), 'Integrated Gradients'),
    ]
    for ax, (imagem, titulo) in zip(linha, paineis):
        ax.imshow(imagem)
        ax.set_title(titulo, fontsize=11)
        ax.axis('off')
plt.suptitle(f'Comparação das explicações — {MODELO_FIG} / {DATASET_FIG}', fontsize=15, fontweight='bold', y=1.0)
salvar(f'painel_xai_{NOME_ARQUIVO}.png', dpi=130)

# Latência e Tamanho em TFLite (todos os modelos anexados)

Mesma metodologia do `mobilenet.ipynb`: conversão com `Optimize.DEFAULT`, uma imagem por vez, 10 execuções de
aquecimento descartadas e média de 100. Serve para preencher a linha que ficou em branco na tabela comparativa
(MobileNetV3Large × BRACOL) — basta anexar o `.keras` dele também.

In [ ]:
N_WARMUP, N_RUNS = 10, 100

if MEDIR_LATENCIA:
    linhas_lat = []
    amostra = next(iter(test_dataset))[0][:1].numpy().astype(np.float32)
    for caminho in MODELOS:
        nome = os.path.splitext(os.path.basename(caminho))[0]
        modelo_nome, _, dataset = nome.partition('_')
        m = tf.keras.models.load_model(caminho, compile=False)
        if m.input_shape[1:3] != amostra.shape[1:3]:
            print(f"  {nome}: entrada {m.input_shape} diferente da amostra — pulando.")
            continue
        conv = tf.lite.TFLiteConverter.from_keras_model(m)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        caminho_tflite = os.path.join('outputs', nome, f'{nome}.tflite')
        os.makedirs(os.path.dirname(caminho_tflite), exist_ok=True)
        with open(caminho_tflite, 'wb') as f:
            f.write(conv.convert())

        interp = tf.lite.Interpreter(model_path=caminho_tflite)
        interp.allocate_tensors()
        entrada = interp.get_input_details()[0]['index']
        for _ in range(N_WARMUP):
            interp.set_tensor(entrada, amostra)
            interp.invoke()
        tempos = []
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            interp.set_tensor(entrada, amostra)
            interp.invoke()
            tempos.append((time.perf_counter() - t0) * 1000)
        linhas_lat.append({'Modelo': modelo_nome, 'Dataset': dataset,
                           'Lat. Exportado (ms)': round(float(np.mean(tempos)), 2),
                           'Lat. Exportado p95 (ms)': round(float(np.percentile(tempos, 95)), 2),
                           'Tamanho Exportado (MB)': round(os.path.getsize(caminho_tflite) / 1024 ** 2, 2)})
        print(f"  {nome}: {linhas_lat[-1]['Lat. Exportado (ms)']} ms | {linhas_lat[-1]['Tamanho Exportado (MB)']} MB")
        del m, interp
        tf.keras.backend.clear_session()

    df_lat = pd.DataFrame(linhas_lat)
    print(df_lat.to_string(index=False))
    df_lat.to_csv(os.path.join('outputs', 'latencia_tflite.csv'), index=False)

# Download

In [ ]:
with zipfile.ZipFile('figuras_modelo.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('outputs/**/*.png', recursive=True) + glob.glob('outputs/**/*.csv', recursive=True):
        z.write(f)
print(f"figuras_modelo.zip: {os.path.getsize('figuras_modelo.zip') / 1024 ** 2:.1f} MB")

from IPython.display import FileLink
FileLink('figuras_modelo.zip')